Save as: module1-simulation-lab.ipynb

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncap/micro-course/blob/main/module-01/module1-simulation-lab.ipynb)

# Lab 1 — Replicating Choice Anomalies with Persona-Varied Agents
*Microeconomic Theory for Experimentalists & Applied Microeconomists*

**Research question:** Do LLM agents reproduce the Allais / certainty-effect
pattern — and does the pattern respond to persona wording and stakes
framing the way human subjects' choices respond to their circumstances?

**The task (Kahneman & Tversky 1979 payoffs):**

- **Q1:** A = \$2400 for sure; B = 33% \$2500, 66% \$2400, 1% \$0
- **Q2:** C = 34% \$2400, 66% \$0; D = 33% \$2500, 67% \$0

Expected utility says your Q1 and Q2 answers must agree (A with C, or B
with D) — PS1 Part A has you prove it. Humans disagree: KT79 found **82%**
chose A and **83%** chose D, so at least **65%** violated EU, all in the
direction the certainty effect predicts.

**Benchmarks (three-way comparison):** the KT79 shares above, AND the
class's own Lecture-1 choices. Your write-up compares literature vs. this
class vs. silicon.

**Hypotheses to state before running (see handout):** H1 modal silicon
pattern = (A, D); H2 violation rate moves with persona; H3 real-payment
framing shifts toward EU-consistency (commit to a direction first).

**How to run:** no prior Python experience needed. Run the cells top to
bottom. Your required modification (handout) goes **only** in the cells
marked **CHANGE HERE**. Course conventions: fixed reported temperature,
persona grid, parse-failure logging, prompt logs saved for your write-up
appendix, robustness section at the end.


## Setup and the DRY_RUN switch

The notebook starts in **DRY_RUN mode**: it executes end-to-end on synthetic
placeholder data, with no API key and no cost, so you can check your design
and see the analysis pipeline work.

> ⚠️ **DRY_RUN output is fake.** It is generated by `mock_response()`, not
> by a model. Never report it as a result.

When your design is ready, set `DRY_RUN = False` in the cell below and run
again in Colab — you will be asked for your Anthropic API key via a hidden
prompt (the key is never stored in the notebook file).


In [ ]:
%pip install anthropic pandas matplotlib --quiet

In [ ]:
import itertools
import json
import random
import re
import time

import pandas as pd

DRY_RUN = True   # <-- set to False in Colab to query the real model

MODEL = "claude-sonnet-5"   # swap models here to test robustness across models
TEMPERATURE = 1.0            # fixed and reported — part of the experimental design
N_PER_CELL = 30              # decisions per persona x treatment cell

if DRY_RUN:
    print("=" * 62)
    print("DRY RUN: synthetic placeholder data — NOT model behavior.")
    print("Set DRY_RUN = False to run the real experiment.")
    print("=" * 62)

    class _MockBlock:
        def __init__(self, text):
            self.text = text

    class _MockResponse:
        def __init__(self, text):
            self.content = [_MockBlock(text)]

    class _MockMessages:
        def create(self, **kwargs):
            # mock_response() is defined in the design cell below
            return _MockResponse(mock_response(kwargs["messages"][0]["content"]))

    class _MockClient:
        messages = _MockMessages()

    client = _MockClient()
else:
    from getpass import getpass

    import anthropic

    # Never hard-code API keys. getpass keeps the key out of the notebook file.
    API_KEY = getpass("Paste your Anthropic API key: ")
    client = anthropic.Anthropic(api_key=API_KEY)


## Experimental design — CHANGE HERE (all four blocks live in this cell)

3 personas × 2 stakes framings = 6 cells × 30 = 180 calls.

Personas are described in **plain behavioral language**, not with utility
parameters. If we wrote "your risk-aversion coefficient is 2", the model
would compute the textbook answer — that tests arithmetic, not behavior.
Describing dispositions and letting behavior emerge is the closer analogue
to a human subject pool.

Your required modification will usually mean **(1)** editing `PAYOFFS` or
adding a factor to `PERSONA_GRID` (e.g. a new persona, or
`"scale": ["x1", "x100"]`), **(2)** weaving it into `build_prompt` *without
changing any other wording across conditions*, and **(3)** touching
`parse_response` / `mock_response` only if your answer format changes
(e.g. the Holt–Laury modification switches to a 10-row format — see the
comment block at the end of this cell).


In [ ]:
PERSONAS = {                                     # CHANGE HERE (1 of 4)
    "baseline": (
        "You are an adult participating in a paid decision study."
    ),
    "cautious": (
        "You are an adult participating in a paid decision study. You are "
        "careful with money: you double-check bills, keep an emergency "
        "fund, and hate the feeling of losing something you already had."
    ),
    "gambler": (
        "You are an adult participating in a paid decision study. You "
        "enjoy betting: you play poker with friends, buy the occasional "
        "lottery ticket, and believe fortune favors the bold."
    ),
}

PERSONA_GRID = {
    "persona": list(PERSONAS),
    "stakes": ["hypothetical", "real_payment"],   # framing treatment
}

# KT79 payoffs. The required "scale the stakes" modification edits these.
PAYOFFS = {"high": 2500, "mid": 2400}


def build_prompt(persona: dict) -> str:          # CHANGE HERE (2 of 4)
    """Two Allais questions, one prompt. Design rules (they matter):
    persona first, task second, response format last; identical wording
    across conditions except the manipulated sentence; no hint that any
    choice is 'consistent' or 'rational'."""
    hi, mid = PAYOFFS["high"], PAYOFFS["mid"]
    if persona["stakes"] == "hypothetical":
        stakes_text = (
            "The amounts below are hypothetical; imagine them as real and "
            "answer as you genuinely would."
        )
    else:
        stakes_text = (
            "One of your two answers will be selected at random and you "
            "will actually be paid according to your choice."
        )

    return (
        f"{PERSONAS[persona['persona']]} {stakes_text}\n\n"
        "Question 1 - choose A or B:\n"
        f"  A: ${mid} for sure.\n"
        f"  B: 33% chance of ${hi}, 66% chance of ${mid}, 1% chance of $0.\n\n"
        "Question 2 - choose C or D:\n"
        f"  C: 34% chance of ${mid}, 66% chance of $0.\n"
        f"  D: 33% chance of ${hi}, 67% chance of $0.\n\n"
        'Respond with ONLY a JSON object: {"q1": "A" or "B", "q2": "C" or "D"}.'
    )


def parse_response(text: str) -> "dict | None":  # CHANGE HERE (3 of 4)
    """Extract the two choices. Return None on failure — never guess."""
    match = re.search(
        r'\{\s*"q1"\s*:\s*"([AB])"\s*,\s*"q2"\s*:\s*"([CD])"\s*\}',
        text, re.IGNORECASE,
    )
    if match:
        return {"q1": match.group(1).upper(), "q2": match.group(2).upper()}
    return None


def mock_response(prompt: str) -> str:           # CHANGE HERE (4 of 4)
    """DRY_RUN only: synthetic placeholder answers so the notebook executes
    without API calls. NOT model behavior — these numbers are made up and
    deliberately mimic the human pattern (certainty effect) so the analysis
    tables are legible. Edit only if your answer format changes."""
    if "fortune favors the bold" in prompt:      # gambler
        p_a, p_d = 0.40, 0.85
    elif "emergency fund" in prompt:             # cautious
        p_a, p_d = 0.92, 0.70
    else:                                        # baseline
        p_a, p_d = 0.80, 0.82
    if "actually be paid" in prompt:             # real-payment framing
        p_a, p_d = min(p_a + 0.05, 0.99), max(p_d - 0.10, 0.01)
    q1 = "A" if random.random() < p_a else "B"
    q2 = "D" if random.random() < p_d else "C"
    return json.dumps({"q1": q1, "q2": q2})


# --- Holt-Laury modification (option 3 in the handout) -----------------------
# Replace the two questions with the 10-row HL list: row k (k = 1..10) offers
#   Option A: k/10 chance of $2.00, else $1.60
#   Option B: k/10 chance of $3.85, else $0.10
# Ask for {"rows": ["A" or "B", ... 10 entries]}, parse the first switch row,
# and benchmark against the human modal switch at rows 5-6 (Holt & Laury 2002).
# Remember to rewrite parse_response AND mock_response for the new format.


## Run the experiment *(do not modify)*

Runs every persona × treatment cell, logs parse failures instead of
dropping them, saves `lab1_results.csv` and `lab1_prompts_log.json`
(required in your write-up appendix). With `DRY_RUN = False` this makes
~180 API calls: expect a couple of minutes and a small API spend.


In [ ]:
def run_experiment() -> pd.DataFrame:
    keys = list(PERSONA_GRID)
    cells = [dict(zip(keys, combo)) for combo in itertools.product(*PERSONA_GRID.values())]
    print(f"{len(cells)} cells x {N_PER_CELL} obs = {len(cells) * N_PER_CELL} decisions")

    rows = []
    for cell in cells:
        prompt = build_prompt(cell)
        for i in range(N_PER_CELL):
            try:
                response = client.messages.create(
                    model=MODEL,
                    max_tokens=100,
                    temperature=TEMPERATURE,
                    messages=[{"role": "user", "content": prompt}],
                )
                raw = response.content[0].text
                decision = parse_response(raw)
            except Exception as err:            # log failures; never drop silently
                raw, decision = f"ERROR: {err}", None
                time.sleep(5)
            rows.append({
                **cell, "rep": i,
                "q1": None if decision is None else decision["q1"],
                "q2": None if decision is None else decision["q2"],
                "raw": raw, "model": MODEL, "temperature": TEMPERATURE,
            })
        done = [r for r in rows if r["q1"] is not None]
        print(f"  cell {cell} done ({len(done)}/{len(rows)} parsed)")

    df = pd.DataFrame(rows)
    df.to_csv("lab1_results.csv", index=False)
    # Save exact prompts — required in the write-up appendix.
    with open("lab1_prompts_log.json", "w") as f:
        json.dump({str(c): build_prompt(c) for c in cells}, f, indent=2)
    return df


df = run_experiment()


## Analysis vs. human benchmarks *(do not modify)*

KT79: 82% chose A, 83% chose D; ≥65% chose the EU-inconsistent pair (A, D).
EU-consistent patterns are (A, C) and (B, D). Enter the class's own
Lecture-1 shares below for the three-way comparison.


In [ ]:
# Human benchmarks. KT79 = Kahneman & Tversky (1979), Problems 1-2.
KT_SHARE_A, KT_SHARE_D = 0.82, 0.83
# Enter your class's Lecture-1 shares here (from the instructor):
CLASS_SHARE_A, CLASS_SHARE_D = None, None

valid = df.dropna(subset=["q1", "q2"]).copy()
print(f"Parse-failure rate: {df['q1'].isna().mean():.3f} "
      "(report this in Section 4 of the write-up)")

valid["chose_A"] = valid["q1"] == "A"
valid["chose_D"] = valid["q2"] == "D"
valid["violation"] = (valid["q1"] + valid["q2"]).isin(["AD", "BC"])

print("\n=== Share choosing A, D, and violating EU, by cell ===")
table = (valid.groupby(["persona", "stakes"])[["chose_A", "chose_D", "violation"]]
         .mean().round(3))
print(table)

print("\n=== Three-way comparison (overall) ===")
print(f"Silicon:    A {valid['chose_A'].mean():.2f} | D {valid['chose_D'].mean():.2f} "
      f"| violation {valid['violation'].mean():.2f}")
print(f"KT79 humans: A {KT_SHARE_A:.2f} | D {KT_SHARE_D:.2f} | violation >= "
      f"{KT_SHARE_A + KT_SHARE_D - 1:.2f}")
if CLASS_SHARE_A is not None:
    print(f"This class:  A {CLASS_SHARE_A:.2f} | D {CLASS_SHARE_D:.2f}")
else:
    print("This class:  (enter CLASS_SHARE_A / CLASS_SHARE_D above)")

print("\nEU-consistent patterns are AC and BD; the certainty effect "
      "predicts the violations pile up on AD, not BC:")
print(valid.groupby(valid["q1"] + valid["q2"]).size())


## Robustness — required in every lab

Rerun at least the `baseline × hypothetical` and `gambler × real_payment`
cells with:

1. **a paraphrased prompt** (rewrite `build_prompt`'s wording, same
   content);
2. **a second model** (change `MODEL` in the setup cell);
3. **the "economist persona"** — add a persona stating the agent is an
   economist. If the violation vanishes, what does that tell you about
   what the baseline result was made of?

Report in Section 4 of the write-up whether your conclusions survive
(1)–(3). If your violation rate dies under paraphrase, it is a fact about
your prompt, not about silicon risk preferences.

**Limits of silicon subjects:** these agents have read Kahneman & Tversky;
they face no real stakes; their "personas" are prompt text, not
preferences. A match with 82/83 is not validation — matching a published
statistic *to the point* is, if anything, evidence of recitation (PS1
Part C takes this head-on). Either way, the deliverable is the mechanism
and the design change it implies for the human study (write-up Section 6).
